In [31]:
from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime
from typing import Union, Optional

from geopy.geocoders import Nominatim
from timezonefinder import TimezoneFinder
from zoneinfo import ZoneInfo

import astropy.units as u
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
from astropy.utils import iers


In [4]:
# Allow precise Earth orientation parameters (falls back gracefully if offline)
iers.conf.auto_download = True

@dataclass
class PlaceTime:
    place_query: str
    display_name: str
    latitude: float
    longitude: float
    timezone: str
    dt_local: datetime      # timezone-aware
    time_astropy: Time      # astropy Time (UTC internally)

def resolve_place_and_time(
    place: str,
    local_dt: Union[str, datetime],
    *,
    geopy_user_agent: str = "astro_utility_app",
) -> PlaceTime:
    """
    Resolve a place name to (lat, lon, timezone), attach timezone to a local datetime,
    and return an astropy Time object.

    - place: e.g. "Stilwell, KS" or "Stilwell Kansas"
    - local_dt: naive or aware datetime, or ISO string (e.g. "2025-09-09 21:00:00")
                If naive, this function assumes it is *local time at the place*.
                If aware, its tzinfo is respected and the place's timezone is still returned.
    """
    # Geocode the place
    geolocator = Nominatim(user_agent=geopy_user_agent)
    loc = geolocator.geocode(place)
    if loc is None:
        raise ValueError(f"Could not geocode place: {place!r}")

    lat = float(loc.latitude)
    lon = float(loc.longitude)
    display_name = loc.address

    # Find IANA timezone name from lat/lon
    tf = TimezoneFinder()
    tzname = tf.timezone_at(lat=lat, lng=lon)
    if tzname is None:
        # Fallback: some rare spots may return None if on water, etc.
        tzname = tf.closest_timezone_at(lat=lat, lng=lon)
    if tzname is None:
        raise RuntimeError(f"Could not determine timezone for coordinates ({lat}, {lon}).")

    # Parse/normalize local datetime
    if isinstance(local_dt, str):
        # Accept flexible ISO-like strings; prefer fromisoformat when possible
        try:
            dt = datetime.fromisoformat(local_dt)
        except ValueError:
            # Very loose fallback parse; you can swap in dateutil.parser if you prefer
            # but keeping stdlib-only here.
            raise ValueError("Provide local_dt as ISO string like 'YYYY-MM-DD HH:MM:SS' or a datetime object.")
    elif isinstance(local_dt, datetime):
        dt = local_dt
    else:
        raise TypeError("local_dt must be a str or datetime.datetime")

    if dt.tzinfo is None:
        # Assume the supplied time is local time *at the place*
        dt_local = dt.replace(tzinfo=ZoneInfo(tzname))
    else:
        # If user passed an aware datetime, keep it as-is
        dt_local = dt

    # Build astropy Time (internally UTC)
    t = Time(dt_local)

    return PlaceTime(
        place_query=place,
        display_name=display_name,
        latitude=lat,
        longitude=lon,
        timezone=tzname,
        dt_local=dt_local,
        time_astropy=t,
    )

# ---------- Optional convenience: compute Alt/Az from RA/Dec (decimal degrees) ----------
def altaz_from_place(
    ra_deg: float,
    dec_deg: float,
    place: str,
    local_dt: Union[str, datetime],
    *,
    elevation_m: float = 0.0,
    pressure_hPa: Optional[float] = 1013.25,  # None disables refraction
    temperature_C: float = 10.0,
    relative_humidity: float = 0.5,
    obs_wavelength_um: float = 0.55,
    geopy_user_agent: str = "astro_utility_app",
):
    """
    Convenience wrapper:
      - Resolves place & local time
      - Builds EarthLocation & AltAz frame with optional refraction
      - Transforms a target (RA/Dec in decimal degrees) to topocentric Alt/Az

    Returns:
      (altaz_coord, meta) where:
        altaz_coord: astropy.coordinates.SkyCoord in AltAz frame
        meta:        PlaceTime dataclass with resolution details
    """
    meta = resolve_place_and_time(place, local_dt, geopy_user_agent=geopy_user_agent)

    location = EarthLocation(lat=meta.latitude * u.deg,
                             lon=meta.longitude * u.deg,
                             height=elevation_m * u.m)

    pressure = 0 * u.hPa if pressure_hPa is None else pressure_hPa * u.hPa
    frame = AltAz(obstime=meta.time_astropy,
                  location=location,
                  pressure=pressure,
                  temperature=temperature_C * u.deg_C,
                  relative_humidity=relative_humidity,
                  obswl=obs_wavelength_um * u.um)

    target = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    altaz = target.transform_to(frame)
    return altaz, meta


In [7]:
# 1) Resolve place + time only
info = resolve_place_and_time("Stilwell, KS", "2025-09-09 21:00:00")
print(f"Resolved: {info.display_name}")
print(f"Lat/Lon:  {info.latitude:.5f}, {info.longitude:.5f}")
print(f"Timezone: {info.timezone}")
print(f"Local dt: {info.dt_local.isoformat()}")
print(f"Astropy UTC: {info.time_astropy.utc.iso}")

# 2) Compute Alt/Az for M31 given RA/Dec in decimal degrees
ra_deg = 10.6847083   # M31
dec_deg = 41.26875
altaz, meta = altaz_from_place(ra_deg, dec_deg, "Stilwell, KS", "2025-09-08 23:22:00",
                                elevation_m=330, pressure_hPa=1013.25)
print(f"Altitude: {altaz.alt.to(u.deg):.3f}")
print(f"Azimuth : {altaz.az.to(u.deg):.3f}")
if hasattr(altaz, "secz"):
    print(f"Airmass : {altaz.secz:.3f}")

Resolved: Stilwell, Aubry, Johnson County, Kansas, United States
Lat/Lon:  38.76918, -94.65635
Timezone: America/Chicago
Local dt: 2025-09-09T21:00:00-05:00
Astropy UTC: 2025-09-10 02:00:00.000
Altitude: 50.948 deg
Azimuth : 69.104 deg
Airmass : 1.288


In [14]:
from astropy import units as u
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from typing import Optional, Union
def altaz_from_lat_long_times(
    ra_deg: float,
    dec_deg: float,
    latitude: float,
    longitude: float,
    local_dt: list[datetime],
    elevation_m: float = 0.0,
    pressure_hPa: Optional[float] = 1013.25,  # None disables refraction
    temperature_C: float = 10.0,
    relative_humidity: float = 0.5,
    obs_wavelength_um: float = 0.55,
) -> Union[SkyCoord, list[SkyCoord]]:

    location = EarthLocation(lat=latitude * u.deg,
                             lon=longitude * u.deg,
                             height=elevation_m * u.m)

    pressure = 0 * u.hPa if pressure_hPa is None else pressure_hPa * u.hPa
    frame = AltAz(obstime=local_dt,
                  location=location,
                  pressure=pressure,
                  temperature=temperature_C * u.deg_C,
                  relative_humidity=relative_humidity,
                  obswl=obs_wavelength_um * u.um)

    target = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    scList = target.transform_to(frame)
    return scList


In [22]:
Lat = 38.76918
Long = -94.65635

ra_deg = 10.6847083   # M31
dec_deg = 41.26875

# start_obs_time = datetime(2025, 9, 8, 21, 0, 0, tzinfo=ZoneInfo("America/Chicago")) # correct
# obs_time = datetime(2025, 9, 8, 23, 22, 0) # WRONG - must be server time??? naive local time
start_obs_time = datetime.now(tz=ZoneInfo("America/Chicago")) # correct current time
# obs_time = datetime.now() # WRONG

obs_times = []
for hour in range(0, 6):
    obs_time = start_obs_time + timedelta(hours=hour)
    obs_times.append(obs_time)

altaz_list = altaz_from_lat_long_times(ra_deg, dec_deg, Lat, Long, obs_times,
                                        elevation_m=330, pressure_hPa=1013.25)
# time format YYYY-MM-DD HH:MM AM/PM
for obs_time, altaz in zip(obs_times, altaz_list):
    if obs_time.tzinfo is None:
        raise ValueError("obs_time must be timezone-aware")
    if altaz is None:
        raise RuntimeError("altaz computation failed")

    if not hasattr(altaz, "alt") or not hasattr(altaz, "az") or not hasattr(altaz, "secz"):
        raise RuntimeError("altaz result missing expected attributes")
    if not isinstance(altaz.alt, u.Quantity) or not isinstance(altaz.az, u.Quantity) or not isinstance(altaz.secz, u.Quantity):
        raise RuntimeError("altaz attributes are not astropy Quantities")

    print(f"At {obs_time.strftime('%Y-%m-%d %I:%M %p')} Altitude: {altaz.alt.deg:.2f}")
    print(f"At {obs_time.strftime('%Y-%m-%d %I:%M %p')} Azimuth : {altaz.az.to(u.deg):.3f}") # type: ignore
    print(f"At {obs_time.strftime('%Y-%m-%d %I:%M %p')} AirMass : {altaz.secz:.2f}")


At 2025-09-16 08:30 PM Altitude: 26.59
At 2025-09-16 08:30 PM Azimuth : 56.836 deg
At 2025-09-16 08:30 PM AirMass : 2.23
At 2025-09-16 09:30 PM Altitude: 36.71
At 2025-09-16 09:30 PM Azimuth : 62.639 deg
At 2025-09-16 09:30 PM AirMass : 1.67
At 2025-09-16 10:30 PM Altitude: 47.34
At 2025-09-16 10:30 PM Azimuth : 67.639 deg
At 2025-09-16 10:30 PM AirMass : 1.36
At 2025-09-16 11:30 PM Altitude: 58.34
At 2025-09-16 11:30 PM Azimuth : 71.702 deg
At 2025-09-16 11:30 PM AirMass : 1.17
At 2025-09-17 12:30 AM Altitude: 69.56
At 2025-09-17 12:30 AM Azimuth : 74.077 deg
At 2025-09-17 12:30 AM AirMass : 1.07
At 2025-09-17 01:30 AM Altitude: 80.77
At 2025-09-17 01:30 AM Azimuth : 69.708 deg
At 2025-09-17 01:30 AM AirMass : 1.01


In [4]:
from datetime import datetime, timezone
from typing import Iterable, Tuple, List
import astronomy
from math import floor

try:
    from zoneinfo import ZoneInfo
except ImportError:
    from backports.zoneinfo import ZoneInfo

def _ra_to_hours(x) -> float:
    if isinstance(x, (int, float)):
        return float(x)
    parts = str(x).strip().split(':')
    if len(parts) == 3:
        hh, mm, ss = parts
        sign = -1 if hh.startswith('-') else 1
        return sign*(abs(float(hh)) + float(mm)/60 + float(ss)/3600)
    elif len(parts) == 2:
        hh, mm = parts
        return float(hh) + float(mm)/60
    else:
        return float(x)

def _dec_to_degs(x) -> float:
    if isinstance(x, (int, float)):
        return float(x)
    parts = str(x).strip().split(':')
    if len(parts) == 3:
        dd, mm, ss = parts
        sign = -1 if dd.startswith('-') else 1
        return sign*(abs(float(dd)) + float(mm)/60 + float(ss)/3600)
    elif len(parts) == 2:
        dd, mm = parts
        return float(dd) + float(mm)/60
    else:
        return float(x)

def datetime_to_days_since_j2000(dt: datetime) -> float:
    if dt.tzinfo is None:
        raise ValueError("dt must be timezone-aware in UTC")
    dt = dt.astimezone(timezone.utc)
    year = dt.year
    month = dt.month
    day = dt.day
    hour = dt.hour
    minute = dt.minute
    second = dt.second + dt.microsecond * 1e-6

    if month <= 2:
        year -= 1
        month += 12
    A = year // 100
    B = 2 - A + (A // 4)
    frac_day = (hour + (minute + second/60.0)/60.0) / 24.0

    JD = (
        floor(365.25 * (year + 4716))
        + floor(30.6001 * (month + 1))
        + day
        + frac_day
        + B
        - 1524.5
    )
    days_since = JD - 2451545.0
    return days_since

def radec_many_to_altaz(
    ra_list: Iterable, dec_list: Iterable,
    lat_deg: float, lon_deg: float,
    dt_local: datetime, tz_name: str,
    *, elevation_m: float = 0.0,
    frame: str = "EQJ",
    refraction: str = "normal"
) -> Tuple[List[float], List[float], datetime, datetime]:
    # Interpret local datetime
    local_tz = ZoneInfo(tz_name)
    if dt_local.tzinfo is None:
        dt_loc_aware = dt_local.replace(tzinfo=local_tz)
    else:
        dt_loc_aware = dt_local.astimezone(local_tz)
    dt_utc = dt_loc_aware.astimezone(timezone.utc)

    # Convert to Astronomy Time
    days = datetime_to_days_since_j2000(dt_utc)
    t = astronomy.Time(days)

    obs = astronomy.Observer(lat_deg, lon_deg, elevation_m)

    refr = astronomy.Refraction.Normal if refraction.lower() == "normal" else astronomy.Refraction.None_

    rot = None
    if frame.upper() == "EQJ":
        rot = astronomy.Rotation_EQJ_EQD(t)
    elif frame.upper() != "EQD":
        raise ValueError("frame must be 'EQJ' or 'EQD'")

    alts: List[float] = []
    azs: List[float] = []

    for ra_val, dec_val in zip(ra_list, dec_list):
        ra_h = _ra_to_hours(ra_val)
        dec_d = _dec_to_degs(dec_val)

        if rot is not None:
            sph_eqj = astronomy.Spherical(lon=ra_h*15.0, lat=dec_d, dist=1.0)
            v_eqj   = astronomy.VectorFromSphere(sph_eqj, t, astronomy.Orientation.EQJ)
            v_eqd   = astronomy.RotateVector(rot, v_eqj)
            sph_eqd = astronomy.SphereFromVector(v_eqd)
            ra_h_eqd = sph_eqd.lon / 15.0
            dec_d_eqd = sph_eqd.lat
        else:
            ra_h_eqd = ra_h
            dec_d_eqd = dec_d

        hor = astronomy.Horizon(t, obs, ra_h_eqd, dec_d_eqd, refr)
        alts.append(hor.altitude)
        azs.append(hor.azimuth)

    return alts, azs, dt_loc_aware, dt_utc



In [5]:
if __name__ == "__main__":
    ras  = ["00:42:44.3",  "05:34:31.9"]      # hours (M31, Rigel approx J2000)
    decs = ["+41:16:09",   "-05:27:00"]       # degrees

    # Stilwell, KS — west longitudes are negative.
    lat, lon, elev = 38.78, -94.66, 300.0

    # Local wall time: Sep 17, 2025 at 9:00 PM America/Chicago
    dt_local = datetime(2025, 9, 17, 21, 0, 0)
    tz = "America/Chicago"

    alts, azs, shown_local, used_utc = radec_many_to_altaz(
        ras, decs, lat, lon, dt_local, tz,
        elevation_m=elev, frame="EQJ", refraction="normal"
    )
    print(shown_local.isoformat(), used_utc.isoformat(), list(zip(alts, azs)))

AttributeError: module 'astronomy' has no attribute 'Orientation'

In [ ]:
from datetime import datetime, timezone
import astronomy

def datetime_to_days_since_j2000(dt: datetime) -> float:
    """Convert UTC datetime → days since J2000.0 (2000-01-01 12:00 UTC)."""
    if dt.tzinfo is None:
        raise ValueError("datetime must be timezone-aware UTC")
    dt = dt.astimezone(timezone.utc)

    y, m, d = dt.year, dt.month, dt.day
    h = dt.hour + dt.minute/60 + (dt.second+dt.microsecond/1e6)/3600

    if m <= 2:
        y -= 1
        m += 12
    A = y // 100
    B = 2 - A + A // 4
    JD = int(365.25*(y+4716)) + int(30.6001*(m+1)) + d + B - 1524.5 + h/24
    return JD - 2451545.0

def radec_to_altaz(ra_hours, dec_deg, obs_lat, obs_lon, dt_utc):
    """Convert RA/Dec (J2000) → Alt/Az for given observer/time."""
    # AE wants east-positive longitude; west = negative.
    obs = astronomy.Observer(obs_lat, obs_lon, 0.0)

    # Time object
    days = datetime_to_days_since_j2000(dt_utc)
    time = astronomy.Time(days)

    # Convert J2000 RA/Dec to EQD (equator-of-date).
    rot = astronomy.Rotation_EQJ_EQD(time)
    eqd = astronomy.RotateEquatorial(rot, ra_hours, dec_deg, 0.0)

    # Horizon from EQD RA/Dec
    hor = astronomy.Horizon(time, obs, eqd.ra, eqd.dec, astronomy.Refraction.None_)
    return hor.altitude, hor.azimuth

# Example: M31 (RA 00h42m44s, Dec +41°16′9″) from Stilwell KS at 2025-09-18 02:00 UTC
dt = datetime(2025, 9, 18, 2, 0, 0, tzinfo=timezone.utc)
alt, az = radec_to_altaz(0.7123, 41.2692, 38.78, -94.66, dt)
print(f"Alt={alt:.2f}°, Az={az:.2f}°")


In [12]:
#!/usr/bin/env python3
"""
RA/DEC to Altitude/Azimuth/Airmass Converter using Astronomy Engine

This script converts right ascension and declination coordinates to 
altitude, azimuth, and air-mass values for a given observer location and time.

Requirements:
    pip install astronomy-engine

Author: Assistant
Python: 3.10+
"""

import astronomy as astro
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from typing import List, Tuple, Dict, Union
import math

def datetime_to_astronomy_time(dt: datetime) -> float:
    """
    Convert a Python datetime object to Astronomy Engine time format.
    
    Astronomy Engine expects time as days since J2000.0 epoch
    (January 1, 2000, 12:00:00 UTC).
    
    Args:
        dt: Python datetime object (should be timezone-aware)
        
    Returns:
        float: Days since J2000.0 epoch
    """
    # J2000.0 epoch: January 1, 2000, 12:00:00 UTC
    j2000_epoch = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
    
    # If datetime is naive, assume it's UTC
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    
    # Calculate the difference and convert to days
    delta = dt - j2000_epoch
    days_since_j2000 = delta.total_seconds() / 86400.0  # 86400 seconds per day
    
    return days_since_j2000

def radec_to_altaz_airmass(
    ra_dec_list: List[Tuple[float, float]], 
    observer_lat: float, 
    observer_lon: float, 
    observation_time: datetime
) -> List[Dict[str, float]]:
    """
    Convert J2000 RA/DEC coordinates to altitude, azimuth, and air-mass.
    
    Astronomy Engine expects J2000 coordinates and automatically handles
    precession, nutation, aberration, and other corrections to compute
    the apparent position at the observation time.
    
    Args:
        ra_dec_list: List of (RA, DEC) tuples in decimal degrees (J2000 epoch)
        observer_lat: Observer latitude in decimal degrees
        observer_lon: Observer longitude in decimal degrees
        observation_time: Timezone-aware datetime object
        
    Returns:
        List of dictionaries containing:
        - 'ra': Right ascension J2000 (degrees)
        - 'dec': Declination J2000 (degrees) 
        - 'altitude': Altitude above horizon (degrees)
        - 'azimuth': Azimuth from north (degrees)
        - 'airmass': Atmospheric air mass (dimensionless)
        - 'visible': Boolean indicating if object is above horizon
    """
    
    # Create observer location
    observer = astro.Observer(observer_lat, observer_lon, 0.0)  # 0.0 = sea level
    
    # Convert datetime to Astronomy Engine time format (days since J2000.0)
    astro_time = astro.Time(datetime_to_astronomy_time(observation_time))
    
    results = []
    
    for ra, dec in ra_dec_list:
        # Convert RA from degrees to hours for Astronomy Engine
        ra_hours = ra / 15.0
        
        # Convert RA/DEC to horizontal coordinates directly
        # Astronomy Engine handles the coordinate transformation internally
        horizontal = astro.Horizon(astro_time, observer, ra_hours, dec, astro.Refraction.Normal)
        
        # Calculate air mass using standard formula: sec(zenith_angle)
        # Air mass is undefined/infinite when altitude <= 0
        zenith_angle = 90.0 - horizontal.altitude
        zenith_radians = math.radians(zenith_angle)
        
        if horizontal.altitude > 0:
            airmass = 1.0 / math.cos(zenith_radians)
            # Apply more accurate airmass formula for low altitudes (Kasten & Young 1989)
            if horizontal.altitude < 20.0:
                airmass = 1.0 / (math.cos(zenith_radians) + 0.50572 * (horizontal.altitude + 6.07995)**(-1.6364))
            visible = True
        else:
            airmass = float('inf')  # Object below horizon
            visible = False
            
        results.append({
            'ra': ra,
            'dec': dec,
            'altitude': horizontal.altitude,
            'azimuth': horizontal.azimuth,
            'airmass': airmass,
            'visible': visible
        })
    
    return results

def single_object_multiple_times(
    ra: float,
    dec: float,
    observer_lat: float, 
    observer_lon: float, 
    datetime_list: List[datetime]
) -> List[Dict[str, Union[float, datetime]]]:
    """
    Calculate altitude, azimuth, and air-mass for a single object at multiple times.
    
    Args:
        ra: Right ascension in decimal degrees (J2000 epoch)
        dec: Declination in decimal degrees (J2000 epoch)
        observer_lat: Observer latitude in decimal degrees
        observer_lon: Observer longitude in decimal degrees
        datetime_list: List of timezone-aware datetime objects
        
    Returns:
        List of dictionaries containing:
        - 'datetime': Original datetime object
        - 'ra': Right ascension J2000 (degrees)
        - 'dec': Declination J2000 (degrees) 
        - 'altitude': Altitude above horizon (degrees)
        - 'azimuth': Azimuth from north (degrees)
        - 'airmass': Atmospheric air mass (dimensionless)
        - 'visible': Boolean indicating if object is above horizon
        - 'j2000_days': Days since J2000.0 for reference
    """
    
    # Create observer location
    observer = astro.Observer(observer_lat, observer_lon, 0.0)  # 0.0 = sea level
    
    # Convert RA from degrees to hours for Astronomy Engine
    ra_hours = ra / 15.0
    
    results = []
    
    for obs_time in datetime_list:
        # Convert datetime to Astronomy Engine time format (days since J2000.0)
        astro_time = astro.Time(datetime_to_astronomy_time(obs_time))
        
        # Convert RA/DEC to horizontal coordinates directly
        horizontal = astro.Horizon(astro_time, observer, ra_hours, dec, astro.Refraction.Normal)
        
        # Calculate air mass using standard formula: sec(zenith_angle)
        zenith_angle = 90.0 - horizontal.altitude
        zenith_radians = math.radians(zenith_angle)
        
        if horizontal.altitude > 0:
            airmass = 1.0 / math.cos(zenith_radians)
            # Apply more accurate airmass formula for low altitudes (Kasten & Young 1989)
            if horizontal.altitude < 20.0:
                airmass = 1.0 / (math.cos(zenith_radians) + 0.50572 * (horizontal.altitude + 6.07995)**(-1.6364))
            visible = True
        else:
            airmass = float('inf')  # Object below horizon
            visible = False
            
        results.append({
            'datetime': obs_time,
            'ra': ra,
            'dec': dec,
            'altitude': horizontal.altitude,
            'azimuth': horizontal.azimuth,
            'airmass': airmass,
            'visible': visible,
            'j2000_days': datetime_to_astronomy_time(obs_time)
        })
    
    return results

def calculate_rise_transit_set_fast(
    ra_dec_list: List[Tuple[float, float]], 
    observer_lat: float, 
    observer_lon: float, 
    reference_date: datetime,
    elevation_meters: float = 0.0
) -> List[Dict[str, Union[float, datetime, str]]]:
    """
    Fast calculation of rise, transit, and set times using Astronomy Engine's built-in functions.
    
    This method uses DefineStar and SearchRiseSet for much faster and more accurate calculations
    than the manual search approach. Uses a single "dso" star that gets redefined for each object.
    
    Args:
        ra_dec_list: List of (RA, DEC) tuples in decimal degrees (J2000 epoch)
        observer_lat: Observer latitude in decimal degrees
        observer_lon: Observer longitude in decimal degrees
        reference_date: Reference date for calculations
        elevation_meters: Observer elevation above sea level in meters
        
    Returns:
        List of dictionaries containing:
        - 'ra': Right ascension J2000 (degrees)
        - 'dec': Declination J2000 (degrees)
        - 'rise_time': Rise datetime (or None if never rises)
        - 'transit_time': Transit datetime
        - 'set_time': Set datetime (or None if never sets)
        - 'circumpolar': True if object never sets
        - 'never_visible': True if object never rises
    """
    
    # Create observer location
    observer = astro.Observer(observer_lat, observer_lon, elevation_meters)
    
    # Convert reference date to Astronomy Engine time
    astro_time = astro.Time(datetime_to_astronomy_time(reference_date))
    
    results = []
    
    for ra, dec in ra_dec_list:
        # Convert RA from degrees to hours for DefineStar
        ra_hours = ra / 15.0
        
        try:
            # Define/redefine the "dso" star with the current RA/DEC coordinates
            # DefineStar expects: name, ra_hours, dec_degrees, distance_parsecs
            astro.DefineStar(astro.Body.Star1, ra_hours, dec, 1000.0)
            
            # Get the dso star body object
            star_body = astro.Body.Star1  # or however the dso body is accessed
            
            # Search for rise time (Direction.Rise)
            rise_event = astro.SearchRiseSet(star_body, observer, astro.Direction.Rise, astro_time, 1, 0.0)
            rise_time = None
            if rise_event is not None:
                # Convert back to Python datetime
                rise_days = rise_event.ut
                rise_j2000_epoch = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
                rise_time = rise_j2000_epoch + timedelta(days=rise_days)
            
            # Search for transit time (hour angle = 0)
            try:
                transit_event = astro.SearchHourAngle(star_body, observer, 0.0, astro_time, 1)
                transit_days = transit_event.time.ut
                transit_j2000_epoch = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
                transit_time = transit_j2000_epoch + timedelta(days=transit_days)
            except:
                transit_time = None
            
            # Search for set time (Direction.Set)
            set_event = astro.SearchRiseSet(star_body, observer, astro.Direction.Set, astro_time, 1, 0.0)
            set_time = None
            if set_event is not None:
                # Convert back to Python datetime
                set_days = set_event.ut
                set_j2000_epoch = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
                set_time = set_j2000_epoch + timedelta(days=set_days)
            
            # Determine object status
            circumpolar = (rise_event is None and set_event is None and transit_time is not None)
            never_visible = (rise_event is None and set_event is None and transit_time is None)
            
            # If we have a transit but no rise/set, check altitude to determine status
            if transit_time and (rise_event is None or set_event is None):
                transit_astro_time = astro.Time(datetime_to_astronomy_time(transit_time))
                try:
                    horizontal = astro.Horizon(transit_astro_time, observer, ra_hours, dec, astro.Refraction.Normal)
                    if horizontal.altitude > 0:
                        circumpolar = True
                        never_visible = False
                    else:
                        circumpolar = False
                        never_visible = True
                except:
                    circumpolar = False
                    never_visible = True
            
        except Exception as e:
            # If anything fails, fall back to None values
            print(f"Exception in rise-set-transit-fast processing RA={ra}, DEC={dec}: {e}")
            rise_time = None
            transit_time = None
            set_time = None
            circumpolar = False
            never_visible = True
        
        results.append({
            'ra': ra,
            'dec': dec,
            'rise_time': rise_time,
            'transit_time': transit_time,
            'set_time': set_time,
            'circumpolar': circumpolar,
            'never_visible': never_visible
        })
    
    return results

def calculate_rise_transit_set(
    ra_dec_list: List[Tuple[float, float]], 
    observer_lat: float, 
    observer_lon: float, 
    reference_date: datetime,
    horizon_altitude: float = -0.5833  # Standard astronomical horizon with refraction
) -> List[Dict[str, Union[float, datetime, str]]]:
    """
    Calculate rise, transit, and set times for a list of RA/DEC coordinates.
    
    Args:
        ra_dec_list: List of (RA, DEC) tuples in decimal degrees (J2000 epoch)
        observer_lat: Observer latitude in decimal degrees
        observer_lon: Observer longitude in decimal degrees
        reference_date: Reference date (calculations done for this date)
        horizon_altitude: Altitude threshold for rise/set (default: -0.5833° for standard horizon)
        
    Returns:
        List of dictionaries containing:
        - 'ra': Right ascension J2000 (degrees)
        - 'dec': Declination J2000 (degrees)
        - 'rise_time': Rise datetime (or None if never rises)
        - 'transit_time': Transit datetime (always calculated)
        - 'set_time': Set datetime (or None if never sets)
        - 'rise_azimuth': Azimuth at rise (degrees from north)
        - 'set_azimuth': Azimuth at set (degrees from north)
        - 'transit_altitude': Altitude at transit (degrees)
        - 'circumpolar': True if object never sets
        - 'never_visible': True if object never rises
    """
    
    # Create observer location
    observer = astro.Observer(observer_lat, observer_lon, 0.0)
    
    results = []
    
    for ra, dec in ra_dec_list:
        # Convert RA from degrees to hours
        ra_hours = ra / 15.0
        
        # Calculate transit time (when object crosses meridian)
        # Transit occurs when local hour angle = 0
        # This happens when LST = RA
        
        # Get reference date at midnight UTC
        ref_midnight = reference_date.replace(hour=0, minute=0, second=0, microsecond=0)
        if ref_midnight.tzinfo is None:
            ref_midnight = ref_midnight.replace(tzinfo=timezone.utc)
        
        transit_time = None
        rise_time = None
        set_time = None
        rise_azimuth = None
        set_azimuth = None
        transit_altitude = None
        
        # Search for transit time by finding when object is closest to meridian
        # We'll check every 6 minutes over 24 hours to find the peak altitude
        best_altitude = -90.0
        
        for minutes in range(0, 24*60, 6):  # Every 6 minutes for 24 hours
            test_time = ref_midnight + timedelta(minutes=minutes)
            astro_time = astro.Time(datetime_to_astronomy_time(test_time))
            
            try:
                horizontal = astro.Horizon(astro_time, observer, ra_hours, dec, astro.Refraction.Normal)
                
                if horizontal.altitude > best_altitude:
                    best_altitude = horizontal.altitude
                    transit_time = test_time
                    transit_altitude = horizontal.altitude
            except:
                continue
        
        # Determine if object is circumpolar or never visible
        # Calculate the object's altitude at upper and lower culmination
        lat_rad = math.radians(observer_lat)
        dec_rad = math.radians(dec)
        
        # Upper culmination altitude
        upper_culmination = math.degrees(math.asin(math.sin(lat_rad) * math.sin(dec_rad) + 
                                                  math.cos(lat_rad) * math.cos(dec_rad)))
        
        # Lower culmination altitude  
        lower_culmination = math.degrees(math.asin(math.sin(lat_rad) * math.sin(dec_rad) - 
                                                  math.cos(lat_rad) * math.cos(dec_rad)))
        
        circumpolar = lower_culmination > horizon_altitude
        never_visible = upper_culmination < horizon_altitude
        
        # If object rises and sets, find those times
        if not circumpolar and not never_visible:
            # Search for rise time (altitude crossing horizon going up)
            for minutes in range(0, 24*60, 3):  # Every 3 minutes
                test_time = ref_midnight + timedelta(minutes=minutes)
                astro_time = astro.Time(datetime_to_astronomy_time(test_time))
                
                try:
                    horizontal = astro.Horizon(astro_time, observer, ra_hours, dec, astro.Refraction.Normal)
                    
                    # Check if we're near the horizon altitude
                    if abs(horizontal.altitude - horizon_altitude) < 0.1:  # Within 0.1 degrees
                        # Check if this is rising (altitude increasing)
                        next_time = test_time + timedelta(minutes=10)
                        next_astro_time = astro.Time(datetime_to_astronomy_time(next_time))
                        next_horizontal = astro.Horizon(next_astro_time, observer, ra_hours, dec, astro.Refraction.Normal)
                        
                        if next_horizontal.altitude > horizontal.altitude and rise_time is None:
                            rise_time = test_time
                            rise_azimuth = horizontal.azimuth
                        elif next_horizontal.altitude < horizontal.altitude and set_time is None and test_time > (transit_time or ref_midnight):
                            set_time = test_time
                            set_azimuth = horizontal.azimuth
                except:
                    continue
        
        results.append({
            'ra': ra,
            'dec': dec,
            'rise_time': rise_time,
            'transit_time': transit_time,
            'set_time': set_time,
            'rise_azimuth': rise_azimuth,
            'set_azimuth': set_azimuth,
            'transit_altitude': transit_altitude,
            'circumpolar': circumpolar,
            'never_visible': never_visible
        })
    
    return results

def print_rise_transit_set_results(results: List[Dict[str, Union[float, datetime, str]]], tz_name="America/Chicago") -> None:
    """Print formatted rise/transit/set results table."""
    print(f"{'RA (°)':<10} {'DEC (°)':<10} {'Rise Time (UTC)':<17} {'Transit Time (UTC)':<17} {'Set Time (UTC)':<17} {'Status':<15}")
    print("-" * 110)
    
    for result in results:
        rt = result['rise_time']
        tt = result['transit_time']
        st = result['set_time']
        
        if rt is not None and isinstance(rt, datetime):
            rt_local = rt.astimezone(ZoneInfo(tz_name))
        else :
            rt_local = None
        if tt is not None and isinstance(tt, datetime):
            tt_local = tt.astimezone(ZoneInfo(tz_name))
        else:
            tt_local = None
        if st is not None and isinstance(st, datetime):
            st_local = st.astimezone(ZoneInfo(tz_name))
        else:
            st_local = None

        rise_str = rt_local.strftime('%H:%M:%S') if rt_local else "---"
        transit_str = tt_local.strftime('%H:%M:%S') if tt_local else "---"
        set_str = st_local.strftime('%H:%M:%S') if st_local else "---"

        # if tz_name:
        #     rise_str = rise_str + f" ({tz_name})"
        #     transit_str = transit_str + f" ({tz_name})"
        #     set_str = set_str + f" ({tz_name})"

        if result['circumpolar']:
            status = "Circumpolar"
        elif result['never_visible']:
            status = "Never visible"
        else:
            status = "Rises & sets"
            
        print(f"{result['ra']:<10.3f} {result['dec']:<10.3f} "
              f"{rise_str:<17} {transit_str:<17} {set_str:<17} {status:<15}")
        
        # Print additional details
        # if result['rise_azimuth'] is not None:
        #     print(f"           Rise Az: {result['rise_azimuth']:.1f}°", end="")
        # if result['set_azimuth'] is not None:
        #     print(f"  Set Az: {result['set_azimuth']:.1f}°", end="")
        # if result['transit_altitude'] is not None:
        #     print(f"  Transit Alt: {result['transit_altitude']:.1f}°", end="")
        print()

def print_time_series_results(results: List[Dict[str, Union[float, datetime]]]) -> None:
    print(f"{'DateTime (UTC)':<20} {'Alt (°)':<10} {'Az (°)':<10} {'Airmass':<10} {'Visible':<10}")
    print("-" * 70)
    
    for result in results:
        airmass_str = f"{result['airmass']:.3f}" if result['airmass'] != float('inf') else "∞"
        visible_str = "Yes" if result['visible'] else "No"
        dt_str = result['datetime'].strftime('%Y-%m-%d %H:%M:%S')
        
        print(f"{dt_str:<20} "
              f"{result['altitude']:<10.3f} {result['azimuth']:<10.3f} "
              f"{airmass_str:<10} {visible_str:<10}")

def print_results(results: List[Dict[str, float]]) -> None:
    """Print formatted results table."""
    print(f"{'RA (°)':<10} {'DEC (°)':<10} {'Alt (°)':<10} {'Az (°)':<10} {'Airmass':<10} {'Visible':<10}")
    print("-" * 70)
    
    for result in results:
        airmass_str = f"{result['airmass']:.3f}" if result['airmass'] != float('inf') else "∞"
        visible_str = "Yes" if result['visible'] else "No"
        
        print(f"{result['ra']:<10.3f} {result['dec']:<10.3f} "
              f"{result['altitude']:<10.3f} {result['azimuth']:<10.3f} "
              f"{airmass_str:<10} {visible_str:<10}")

# Example usage
if __name__ == "__main__":
    from datetime import timezone, timedelta
    
    # Example coordinates (some bright stars and objects)
    coordinates = [
        (83.633, 22.014),    # Aldebaran
        (279.234, 38.784),   # Vega  
        (310.358, 45.280),   # Deneb
        (201.298, -11.161),  # Spica
        (213.915, 19.182),   # Arcturus
        (37.0, 89.0),         # Polaris
        (123.0, -60.0),      # Crux (not visible from northern hemisphere)
    ]
    
    # Observer location (example: Kitt Peak Observatory)
    latitude = 38.76918    # degrees N
    longitude = -94.65635 # degrees W
    
    # Current time (make it timezone-aware)
    # Using UTC for this example
    obs_time = datetime.now(tz=ZoneInfo("America/Chicago")) #datetime(2024, 6, 15, 22, 30, 0, tzinfo=timezone.utc)
    
    print("=== EXAMPLE 1: Multiple objects at one time ===")
    print("RA/DEC to Altitude/Azimuth/Airmass Conversion")
    print("Input coordinates: J2000 epoch")
    print(f"Observer: {latitude:.4f}° N, {longitude:.4f}° W")
    print(f"Time: {obs_time}")
    print(f"J2000 days: {datetime_to_astronomy_time(obs_time):.6f}")
    print()
    
    # Perform conversion
    results = radec_to_altaz_airmass(coordinates, latitude, longitude, obs_time)
    
    # Print results
    print_results(results)
    
    print("\n" + "="*80)
    print("=== EXAMPLE 2: Single object at multiple times ===")
    
    # Single object: Vega
    vega_ra, vega_dec = 279.234, 38.784
    
    # Create time series (every 2 hours for 24 hours)
    base_time = obs_time #datetime(2024, 6, 15, 18, 0, 0, tzinfo=timezone.utc)
    time_series = [base_time + timedelta(hours=i*2) for i in range(13)]  # 0, 2, 4, ..., 24 hours
    
    print(f"Object: Vega (RA={vega_ra}°, DEC={vega_dec}°)")
    print(f"Observer: {latitude:.4f}° N, {longitude:.4f}° W")
    print(f"Time range: {time_series[0]} to {time_series[-1]}")
    print()
    
    # Calculate for single object over time
    time_results = single_object_multiple_times(vega_ra, vega_dec, latitude, longitude, time_series)
    
    # Print time series results
    print_time_series_results(time_results)
    
    print("\n" + "="*80)
    print("=== EXAMPLE 3: Rise/Transit/Set times ===")
    
    # Calculate rise/transit/set for the same objects
    reference_date = obs_time # datetime(2024, 6, 15, tzinfo=timezone.utc)
    
    print(f"Rise/Transit/Set times for {reference_date.strftime('%Y-%m-%d')}")
    print(f"Observer: {latitude:.4f}° N, {longitude:.4f}° W")
    print()
    
    # Calculate rise/transit/set times (using fast method)
    rts_results = calculate_rise_transit_set_fast(coordinates, latitude, longitude, reference_date)
    
    # Print rise/transit/set results
    print_rise_transit_set_results(rts_results)
    
    print("\nNotes:")
    print("- Fast method uses Astronomy Engine's built-in SearchRiseSet and SearchHourAngle")
    print("- Much more accurate and faster than manual search methods")
    print("- Automatically handles atmospheric refraction and other corrections")
    
    print("\nNotes:")
    print("- Input coordinates are J2000 epoch (Astronomy Engine handles precession/nutation)")
    print("- Time converted to days since J2000.0 epoch (Jan 1, 2000, 12:00 UTC)")
    print("- Airmass ∞ indicates the object is below the horizon")
    print("- Airmass formula includes atmospheric refraction correction")
    print("- Azimuth is measured from North (0°) toward East (90°)")
    print("- Altitude is elevation above the horizon")

=== EXAMPLE 1: Multiple objects at one time ===
RA/DEC to Altitude/Azimuth/Airmass Conversion
Input coordinates: J2000 epoch
Observer: 38.7692° N, -94.6564° W
Time: 2025-09-17 19:42:45.131525-05:00
J2000 days: 9391.529689

RA (°)     DEC (°)    Alt (°)    Az (°)     Airmass    Visible   
----------------------------------------------------------------------
83.633     22.014     -28.108    10.135     ∞          No        
279.234    38.784     85.321     87.938     1.003      Yes       
310.358    45.280     61.935     64.486     1.133      Yes       
201.298    -11.161    6.792      249.893    7.937      Yes       
213.915    19.182     35.582     266.784    1.719      Yes       
37.000     89.000     38.230     1.058      1.616      Yes       
123.000    -60.000    -61.519    211.606    ∞          No        

=== EXAMPLE 2: Single object at multiple times ===
Object: Vega (RA=279.234°, DEC=38.784°)
Observer: 38.7692° N, -94.6564° W
Time range: 2025-09-17 19:42:45.131525-05:00 to 2025